# mod-BAM Output Verification

验证 `write_mod_bam` 输出的BAM文件：
1. MM/ML tag格式是否正确
2. `load_read_results()` 是否能正确解析
3. p_mod_hmm值是否在uint8量化误差范围内保持
4. native/IVT reads的RG tag是否正确
5. samtools / modkit兼容性检查

In [ ]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pysam

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)

TESTDATA = Path("../testdata").resolve()

NATIVE_BAM   = TESTDATA / "native_0.bam"
NATIVE_FASTQ = TESTDATA / "native_0" / "fastq" / "pass.fq.gz"
NATIVE_BLOW5 = TESTDATA / "native_0" / "blow5" / "nanopore.blow5"

IVT_BAM   = TESTDATA / "control_0.bam"
IVT_FASTQ = TESTDATA / "control_0" / "fastq" / "pass.fq.gz"
IVT_BLOW5 = TESTDATA / "control_0" / "blow5" / "nanopore.blow5"

REF_FASTA = TESTDATA / "ref.fa"

for label, path in [
    ("native BAM",   NATIVE_BAM),
    ("native FASTQ", NATIVE_FASTQ),
    ("native BLOW5", NATIVE_BLOW5),
    ("IVT BAM",      IVT_BAM),
    ("IVT FASTQ",    IVT_FASTQ),
    ("IVT BLOW5",    IVT_BLOW5),
    ("ref FASTA",    REF_FASTA),
]:
    status = "OK" if path.exists() else "MISSING"
    print(f"  {status:8s}  {label:15s}  {path}")

## 1. Run pipeline to get HMM results

In [ ]:
from baleen import run_pipeline_streaming

hmm_results, sites, meta = run_pipeline_streaming(
    native_bam=NATIVE_BAM,
    native_fastq=NATIVE_FASTQ,
    native_blow5=NATIVE_BLOW5,
    ivt_bam=IVT_BAM,
    ivt_fastq=IVT_FASTQ,
    ivt_blow5=IVT_BLOW5,
    ref_fasta=REF_FASTA,
    min_depth=1,
    threads=4,
    run_hmm=True,
)

print(f"Contigs: {list(hmm_results.keys())}")
for contig, cmr in hmm_results.items():
    n_pos = len(cmr.position_stats)
    print(f"  {contig}: {n_pos} positions")

## 2. Write mod-BAM with `write_mod_bam`

In [ ]:
import tempfile
from baleen.eventalign._read_bam import write_mod_bam

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="modbam_test_"))
OUTPUT_BAM = OUTPUT_DIR / "read_results.bam"

result_path = write_mod_bam(
    hmm_results,
    native_bam=NATIVE_BAM,
    ivt_bam=IVT_BAM,
    ref_fasta=REF_FASTA,
    output_path=OUTPUT_BAM,
)

print(f"Output BAM:   {result_path}")
print(f"BAM exists:   {result_path.exists()}")
print(f"Index exists: {Path(str(result_path) + '.bai').exists()}")

## 3. Inspect raw BAM records — MM/ML tags

In [ ]:
with pysam.AlignmentFile(str(OUTPUT_BAM), "rb") as bam:
    records = list(bam.fetch())

print(f"Total reads in mod-BAM: {len(records)}")
print()

n_with_mm = sum(1 for r in records if r.has_tag("MM"))
n_with_ml = sum(1 for r in records if r.has_tag("ML"))
n_native  = sum(1 for r in records if r.get_tag("RG") == "native")
n_ivt     = sum(1 for r in records if r.get_tag("RG") == "ivt")

print(f"Reads with MM tag: {n_with_mm}")
print(f"Reads with ML tag: {n_with_ml}")
print(f"Native reads (RG=native): {n_native}")
print(f"IVT reads (RG=ivt):       {n_ivt}")

In [ ]:
# Inspect a few reads in detail
print("=== Sample reads ===")
for r in records[:3]:
    print(f"\nRead: {r.query_name}")
    print(f"  Contig:    {r.reference_name}")
    print(f"  Start:     {r.reference_start}")
    print(f"  CIGAR:     {r.cigarstring}")
    print(f"  Seq len:   {r.query_length}")
    print(f"  RG:        {r.get_tag('RG')}")
    if r.has_tag("MM"):
        mm = r.get_tag("MM")
        ml = r.get_tag("ML")
        print(f"  MM:        {mm[:80]}{'...' if len(mm) > 80 else ''}")
        print(f"  ML (first 10): {list(ml[:10])}")
        print(f"  # tagged positions: {len(ml)}")
    else:
        print("  (no MM/ML tags)")

## 4. Round-trip via `load_read_results`

In [ ]:
from baleen.eventalign._read_bam import load_read_results

df = load_read_results(OUTPUT_BAM)

print(f"Total (read, position) entries: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Unique reads: {df['read_name'].nunique()}")
print(f"Native entries: {df['is_native'].sum()}")
print(f"IVT entries: {(~df['is_native']).sum()}")
print()
print(df.head(10))

## 5. Verify p_mod_hmm values (uint8 quantization check)

In [ ]:
from collections import defaultdict

# Build ground truth from HMM results
ground_truth = {}  # (read_name, contig, position) -> p_mod_hmm

for contig, cmr in hmm_results.items():
    for pos, ps in cmr.position_stats.items():
        for i, name in enumerate(ps.native_read_names):
            p = float(ps.p_mod_hmm[i])
            if not np.isnan(p):
                ground_truth[(name, contig, pos)] = p
        for j, name in enumerate(ps.ivt_read_names):
            p = float(ps.p_mod_hmm[ps.n_native + j])
            if not np.isnan(p):
                ground_truth[(name, contig, pos)] = p

# Compare
errors = []
matched = 0
for _, row in df.iterrows():
    key = (row["read_name"], row["contig"], row["position"])
    if key in ground_truth:
        gt = ground_truth[key]
        loaded = row["p_mod_hmm"]
        err = abs(gt - loaded)
        errors.append(err)
        matched += 1

errors = np.array(errors)
print(f"Matched entries: {matched}/{len(df)}")
print(f"Max quantization error: {errors.max():.6f}")
print(f"Mean quantization error: {errors.mean():.6f}")
print(f"99th percentile error: {np.percentile(errors, 99):.6f}")
print()

# uint8 quantization max error should be <= 1/255 ≈ 0.00392
max_expected = 1.0 / 255.0
if errors.max() <= max_expected + 1e-9:
    print(f"OK: all errors within uint8 quantization bound ({max_expected:.6f})")
else:
    print(f"WARNING: some errors exceed uint8 bound!")

## 6. Per-read tag distribution

In [ ]:
import matplotlib.pyplot as plt

positions_per_read = df.groupby("read_name").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: positions per read
axes[0].hist(positions_per_read, bins=30, edgecolor="white")
axes[0].set_xlabel("# annotated positions per read")
axes[0].set_ylabel("# reads")
axes[0].set_title("MM/ML tag density per read")

# Right: p_mod_hmm distribution by group
native_p = df[df["is_native"]]["p_mod_hmm"]
ivt_p = df[~df["is_native"]]["p_mod_hmm"]

axes[1].hist(native_p, bins=30, alpha=0.6, label=f"native (n={len(native_p)})", edgecolor="white")
axes[1].hist(ivt_p, bins=30, alpha=0.6, label=f"ivt (n={len(ivt_p)})", edgecolor="white")
axes[1].set_xlabel("p_mod_hmm")
axes[1].set_ylabel("count")
axes[1].set_title("p_mod_hmm distribution from mod-BAM")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. samtools compatibility check

In [ ]:
import subprocess

# Check samtools can read MM tags
try:
    result = subprocess.run(
        ["samtools", "view", "-h", str(OUTPUT_BAM)],
        capture_output=True, text=True, timeout=10,
    )
    lines = result.stdout.strip().split("\n")
    mm_lines = [l for l in lines if "MM:Z:" in l]
    print(f"samtools view: {len(lines)} lines total, {len(mm_lines)} with MM:Z tag")
    if mm_lines:
        print(f"\nExample line (truncated):")
        print(mm_lines[0][:200] + "...")
except FileNotFoundError:
    print("samtools not found on PATH — skipping")

In [ ]:
# Check modkit compatibility (if available)
try:
    result = subprocess.run(
        ["modkit", "summary", str(OUTPUT_BAM)],
        capture_output=True, text=True, timeout=30,
    )
    print("modkit summary output:")
    print(result.stdout[:2000] if result.stdout else "(no stdout)")
    if result.stderr:
        print(f"\nstderr: {result.stderr[:500]}")
except FileNotFoundError:
    print("modkit not found on PATH — skipping")

## 8. Cleanup

In [ ]:
import shutil
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
print(f"Cleaned up {OUTPUT_DIR}")